In [4]:
import pandas as pd

df = pd.read_csv("O3.csv")

# pastikan kolom tanggal valid
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ambil hanya bulan dan tahun
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

new_df = pd.DataFrame({
    "date": df['date'],
    "O3": df['O3']
})

new_df.to_csv("O3_Timeseries.csv", index=False)

In [5]:
df = pd.read_csv("O3_Timeseries.csv")
missing_value = df['O3'].isna().sum()
print(missing_value)

6


In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("SO2_Timeseries.csv")
df_clean = df.dropna(subset=['SO2']).copy()

model = IsolationForest(contamination=0.05, random_state=42) # contamination 0.05 = 5%
pred = model.fit_predict(df_clean[['SO2']])

# Nilai -1 merepresentasikan outlier
jumlah_outlier = (pred == -1).sum()
print("Jumlah outlier model Outliers:", jumlah_outlier)

Jumlah outlier model Outliers: 16


In [2]:
import pandas as pd

df_o3 = pd.read_csv("O3_Timeseries.csv")
df_co = pd.read_csv("CO_Timeseries.csv")
df_no2 = pd.read_csv("NO2_Timeseries.csv")
df_so2 = pd.read_csv("SO2_Timeseries.csv")

dataframe_merged = pd.DataFrame({
    "date": df_o3['date'],
    "O3": df_o3['O3'],
    "CO": df_co['CO'],
    "NO2": df_no2['NO2'],
    "SO2": df_so2['SO2']
})

dataframe_merged.to_csv("Polutan_Nganjuk.csv", index=False)

In [3]:
import pandas as pd

df = pd.read_csv("NO2_Baron.csv")

# pastikan kolom tanggal valid
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ambil hanya bulan dan tahun
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

new_df = pd.DataFrame({
    "date": df['date'],
    "NO2": df['NO2']
})

new_df.to_csv("NO2_Baron_1.csv", index=False)

In [12]:
import pandas as pd

df = pd.read_csv("NO2_Baron_Interpolated.csv")

# Pastikan urut berdasarkan tanggal (penting sebelum interpolasi time series!)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Interpolasi nilai NaN pada kolom NO2
df['NO2_filled'] = df['NO2'].interpolate(
    method='linear', limit_direction='both'
)

print(df.head(10))

        date       NO2  NO2_filled
0 2025-08-30  0.000012    0.000012
1 2025-08-31  0.000012    0.000012
2 2025-09-01  0.000019    0.000019
3 2025-09-02  0.000026    0.000026
4 2025-09-03  0.000031    0.000031
5 2025-09-04  0.000031    0.000031
6 2025-09-05  0.000030    0.000030
7 2025-09-06  0.000019    0.000019
8 2025-09-07  0.000026    0.000026
9 2025-09-08  0.000033    0.000033


In [13]:
new_df = pd.DataFrame({
    "date": df['date'],
    "NO2": df['NO2_filled']
})

new_df.to_csv("NO2_Baron_Interpolated.csv", index=False)

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("NO2_Baron_1.csv")
df['date'] = pd.to_datetime(df['date'])

# Hitung IQR
Q1 = df['NO2'].quantile(0.25)
Q3 = df['NO2'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter outlier
outliers_iqr = df[(df['NO2'] < lower_bound) | (df['NO2'] > upper_bound)]

print("Jumlah Outlier (IQR):", len(outliers_iqr))
print(outliers_iqr[['date', 'NO2']].head())

Jumlah Outlier (IQR): 4
          date       NO2
163 2026-02-06  0.000005
218 2026-04-15  0.000059
293 2026-01-30 -0.000004
303 2026-04-20  0.000063


In [15]:
# Tandai outlier menjadi NaN
df['NO2_cleaned'] = df['NO2'].mask(
    (df['NO2'] < lower_bound) | (df['NO2'] > upper_bound)
)

print("Jumlah nilai outlier:", df['NO2_cleaned'].isna().sum())

# Interpolasi linear
df['NO2_filled'] = df['NO2_cleaned'].interpolate(method='linear')
df['NO2_filled'] = df['NO2_filled'].bfill().ffill()

print("Jumlah missing setelah interpolasi:", df['NO2_filled'].isna().sum())

Jumlah nilai outlier: 182
Jumlah missing setelah interpolasi: 0
